# TF-IDF

TF–IDF stands for Term Frequency–Inverse Document Frequency. It is a weighting method that represents a document using word or n-gram features, but gives higher weight to terms that are frequent in that specific document and relatively rare across the full corpus.

In short: Bag of Words counts terms; **TF–IDF weights their importance**.

## Core intuition
### Suppose you have thousands of movie reviews:

- Words such as the, is, and movie may appear in almost every review.

- A word such as masterpiece, predictable, or the bigram waste time may appear in comparatively few reviews.

- TF–IDF reduces the influence of terms that occur in many documents and gives more importance to terms that help distinguish one document from others.

- A term receives a high TF–IDF score when it is:

    - Frequent in a particular document.

    - Uncommon across the wider document collection.



## Formula
For a term t, document d, and corpus D:

- **tfidf(t, d, D) = tf(t, d) × idf(t, D)**

    - Term Frequency (TF)

        - TF measures how often a term occurs in a document: tf(t, d) = count of t in d
            
        - A normalized alternative is: **tf(t, d) = total terms in d / count of t in d**
​
        - Higher TF means the term occurs more often in that document.

    - Inverse Document Frequency (IDF)

        - IDF measures how uncommon the term is across the corpus:

        - **idf(t, D) = log(df(t) / N)** ; Where: N = total number of documents, df(t) = number of documents containing term t.

        - If a term appears in almost all documents, its IDF is low. If it appears in few documents, its IDF is high.

- Scikit-learn’s unsmoothed formulation is:

    - **idf(t) = log( df(t) / n) + 1** ; where n is the total number of documents.

### Small example
- Consider three documents:

    - Doc 1: machine learning is fun
    - Doc 2: machine learning is useful
    - Doc 3: deep learning is fun

| Term     | Documents containing it | IDF intuition                  |
| -------- | ----------------------- | ------------------------------ |
| learning | 3                       | Very low: it occurs everywhere |
| machine  | 2                       | Moderate                       |
| fun      | 2                       | Moderate                       |
| deep     | 1                       | High: it is distinctive        |
| useful   | 1                       | High: it is distinctive        |

- So:

    - In Doc 3, deep gets a high TF–IDF score because it is present there and absent from the other two documents.

    - learning gets a low score even if it appears once in every document, because it does not distinguish documents.

That is why TF–IDF is often better than raw BoW counts for retrieval and classical ML.

## TF–IDF vs Bag of Words

| Aspect                  | Bag of Words                    | TF–IDF                                                 |
| ----------------------- | ------------------------------- | ------------------------------------------------------ |
| Feature value           | Raw count or binary presence    | Importance weight                                      |
| Common terms            | May receive high counts         | Down-weighted if common across documents               |
| Rare, distinctive terms | Counted normally                | Often receive higher weight                            |
| Representation          | Sparse count matrix             | Sparse weighted matrix                                 |
| Word order              | Ignored unless n-grams are used | Ignored unless n-grams are used                        |
| Typical purpose         | Basic baseline, count analysis  | Classification, search, similarity, keyword extraction |

TF–IDF is essentially a weighted Bag of Words. It keeps the same vocabulary-style representation but replaces counts with corpus-aware importance scores.

## Using TfidfVectorizer

Since you are already using scikit-learn’s CountVectorizer, TfidfVectorizer will feel familiar. It is effectively equivalent to CountVectorizer followed by TfidfTransformer.

In [1]:
import pandas as pd

In [2]:
messages = pd.read_csv('data/sms_spam_collection/sms_collection', sep='\t',names=['label', 'message'])

In [3]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

In [5]:
lemmatizer = WordNetLemmatizer()

In [6]:
def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}

    return tag_dict.get(tag, wordnet.NOUN)

In [7]:
# go thru each message -> remove non-alphabetic characters -> lowercase -> tokenize -> remove stopwords -> lemmatize -> join back to string
corpus = []
for i in range(len(messages)):
    message = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    message = message.lower()
    message = nltk.word_tokenize(message)
    tagged = nltk.pos_tag(message)
    message = [lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag)) for word, tag in tagged if word not in set(stopwords.words('english'))]
    corpus.append(' '.join(message))

In [8]:
corpus[:5]

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf life around though']

## Create TF-IDF And NGrams

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [10]:
vectorizer = TfidfVectorizer(max_features=250)

In [29]:
X = vectorizer.fit_transform(corpus).toarray()
print(vectorizer.get_feature_names_out()[:40])

['account' 'already' 'also' 'always' 'amp' 'another' 'anything' 'around'
 'ask' 'awarded' 'babe' 'back' 'best' 'better' 'big' 'bit' 'box' 'boy'
 'bt' 'buy' 'call' 'camera' 'cant' 'car' 'care' 'cash' 'chance' 'chat'
 'check' 'claim' 'class' 'co' 'com' 'come' 'coming' 'contact' 'cool'
 'cost' 'could' 'customer']


In [26]:
not_zero = [X[0][i].round(3) for i in range(len(X[0])) if X[0][i] != 0]
not_zero


[np.float64(0.363),
 np.float64(0.381),
 np.float64(0.455),
 np.float64(0.46),
 np.float64(0.552)]

## With word n-grams

In [37]:
vectorizer = TfidfVectorizer(
    max_features=250,
    ngram_range=(1, 2),
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.90
)

X = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out()[:40])

['account' 'actually' 'age' 'ah' 'aight' 'amp' 'ask' 'awarded' 'babe'
 'baby' 'bad' 'bed' 'best' 'better' 'big' 'birthday' 'bit' 'box' 'boy'
 'bt' 'buy' 'called' 'camera' 'car' 'care' 'cash' 'chance' 'chat' 'check'
 'claim' 'class' 'com' 'come' 'coming' 'contact' 'cool' 'cost' 'customer'
 'da' 'dat']


In [38]:
vectorizer = TfidfVectorizer(
    max_features=250,
    ngram_range=(2, 3),
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.90
)

X = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out()[:40])

['account statement' 'account statement redeemed' 'admirer looking'
 'admirer looking make' 'anytime network' 'anytime network min'
 'attempt contact' 'await collection' 'await collection sae'
 'awarded bonus' 'awarded cd' 'bonus caller' 'bonus caller prize'
 'bt national' 'bt national rate' 'caller prize' 'caller prize nd'
 'camcorder reply' 'camera phone' 'cash await' 'cash await collection'
 'cash prize' 'chance win' 'claim code' 'claim ur' 'claim valid'
 'claim valid hr' 'code expires' 'collection sae' 'come home'
 'come tomorrow' 'contact reveal' 'contact reveal think' 'contact today'
 'contact today draw' 'customer service' 'customer service representative'
 'dating service' 'decimal gt' 'dont know']


In [46]:
vectorizer.vocabulary_['prize claim easy']

np.int64(164)

## Important parameters

| Parameter            | Purpose                                                                                              |
| -------------------- | ---------------------------------------------------------------------------------------------------- |
| ngram_range=(1, 2)   | Use unigrams and bigrams                                                                             |
| min_df=2             | Keep a term only if it occurs in at least two documents                                              |
| max_df=0.90          | Remove terms appearing in more than 90% of documents                                                 |
| max_features=10_000  | Cap vocabulary size                                                                                  |
| stop_words="english" | Remove scikit-learn’s built-in English stop-word list                                                |
| use_idf=True         | Apply inverse document frequency weighting                                                           |
| smooth_idf=True      | Smooth the IDF calculation so unseen/edge cases are handled safely                                   |
| sublinear_tf=True    | Use 1+log⁡(TF)1 + \\log(\\text{TF})1+log(TF) rather than raw TF, reducing the impact of repeated words |
| norm="l2"            | Normalize each document vector to unit L2 length; this is the default                                |

In [47]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(2, 2),
    min_df=1,
    max_df=1.0,
    max_features=1000,
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
    norm="l2"
)

In [48]:
X = vectorizer.fit_transform(corpus).toarray()

print(vectorizer.get_feature_names_out()[:40])

['accept day' 'account statement' 'admirer looking' 'aft ur'
 'afternoon love' 'aight text' 'amp miracle' 'amp need' 'amp sweet'
 'angry childish' 'angry wid' 'ansr sp' 'anytime network' 'apartment went'
 'ard smth' 'arrive customer' 'asap box' 'askd lunch' 'asked dating'
 'attempt contact' 'attempt reach' 'auction send' 'await collection'
 'awarded bonus' 'awarded cd' 'awarded prize' 'awarded sipix' 'babe love'
 'bad day' 'balance currently' 'bcums angry' 'beautiful truth'
 'bedroom minute' 'best life' 'best tone' 'birthday dear' 'birthday wish'
 'blood send' 'blue bay' 'blue blue']


Scikit-learn describes TF–IDF as term frequency multiplied by inverse document frequency, and its vectorizer converts raw document collections into TF–IDF feature matrices.

## fit_transform vs transform
This distinction matters when you train/test split data:

In [55]:
len(corpus)

5572

In [57]:
train_text = corpus[:4000]
test_text = corpus[4000:]

In [58]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(2, 2),
    min_df=1,
    max_df=1.0,
    max_features=1000,
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
    norm="l2"
)

In [59]:
X_train = vectorizer.fit_transform(corpus).toarray()
X_test = vectorizer.transform(test_text).toarray()

- fit_transform(train_texts) learns the vocabulary and IDF values only from training data, then transforms it.

- transform(test_texts) applies the already learned vocabulary and IDF values to test data.

- Do not call fit_transform() separately on test data, because that leaks test-corpus statistics into the training pipeline and yields incompatible feature spaces.

# Advantages of TF-IDF:

- Highlights terms that are distinctive within a document, while down-weighting widely shared terms.

- Simple, interpretable, and efficient for classical NLP pipelines.

- Often strong for document search, similarity, keyword extraction, spam detection, sentiment analysis, and linear text classifiers.

- Works naturally with word n-grams, allowing useful phrases such as not good or machine learning to receive feature weights.

# Limitations

- It remains sparse and high-dimensional, especially with large vocabularies and n-grams.

- It does not understand context, syntax, word order beyond explicit n-grams, or semantic similarity. car and automobile are unrelated features.

- A word unseen during fitting is out-of-vocabulary and cannot contribute a feature at transform time.

- IDF depends on the corpus; the same word can get different importance weights in different document collections.

> OHE → represents one token/category

> BoW → counts vocabulary items in a document

> BoW + n-grams → counts words and short phrases

> TF–IDF → weights those word/phrase features by corpus-wide importance